# 7. Varlık Tabanlı Duygu Analizi (Aspect-Based Sentiment Analysis)

Bu bölümde, önceden eğittiğimiz modelleri (TF-IDF ve Logistic Regression) kullanarak yorumları **Food, Service, Ambience, Price** gibi spesifik alanlara göre parçalıyor ve duygu analizi gerçekleştiriyoruz.

ÖNEMLİ: Bu aşamada dışarıdan hiçbir hazır model (BERT vb.) kullanılmamakta, tamamen 4. ve 5. notebook'ta sıfırdan eğittiğimiz modeller ile çıkarım (inference) yapılmaktadır.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import nltk
import scipy.sparse
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
nltk.download('punkt', quiet=True)
from nltk.tokenize import sent_tokenize


## 7.1 Verilerin ve Modellerin Yüklenmesi

In [ ]:
print("Modeller yükleniyor...")
tfidf_vectorizer = joblib.load('models/tfidf_vectorizer.pkl')
scaler = joblib.load('models/scaler.pkl')
lr_model = joblib.load('models/lr_model.pkl')
print("Modeller başarıyla yüklendi!")

print("\nVeri seti yükleniyor (Sadece test amaçlı 10.000 rastgele satır)...")
df = pd.read_csv('data/reviews_preprocessed.csv').dropna(subset=['cleaned_text'])
df = df.sample(n=min(10000, len(df)), random_state=42).reset_index(drop=True)
print(f"Örnek veri boyutu: {df.shape}")


## 7.2 Aspect (Varlık) Sözlüğünün Tanımlanması

In [ ]:
aspect_keywords = {
    'Food': ['food', 'meal', 'delicious', 'taste', 'chicken', 'meat', 'burger', 'pizza', 'cold', 'hot', 'fresh', 'tasty', 'flavor'],
    'Service': ['service', 'waiter', 'staff', 'manager', 'rude', 'friendly', 'slow', 'fast', 'wait', 'hour', 'minutes', 'table'],
    'Ambience': ['place', 'atmosphere', 'vibe', 'clean', 'dirty', 'loud', 'quiet', 'music', 'environment', 'decor'],
    'Price': ['price', 'cost', 'expensive', 'cheap', 'worth', 'value', 'bill', 'money', 'pay']
}

def get_aspect(sentence):
    sentence_lower = sentence.lower()
    detected_aspects = []
    for aspect, keywords in aspect_keywords.items():
        if any(keyword in sentence_lower for keyword in keywords):
            detected_aspects.append(aspect)
    return detected_aspects


## 7.3 Cümlelere Bölme (Sentence Tokenization) ve Çıkarım

In [ ]:
# Modellerimiz orijinal olarak 8 sayısal özellik + TF-IDF ile eğitildiği için
# cümle tahmininde cümlenin ait olduğu orijinal incelemenin sayısal özelliklerini kullanıyoruz.
numeric_features_cols = [
    'review_length', 'word_count', 'exclamation_count', 'question_count',
    'avg_word_length', 'uppercase_ratio', 'sentiment_polarity', 'sentiment_subjectivity'
]
X_numeric_scaled = scaler.transform(df[numeric_features_cols])

aspect_results = []

print("Cümleler analiz ediliyor ve modeller üzerinden tahminleme yapılıyor...")
for idx, row in df.iterrows():
    review = row['cleaned_text']
    sentences = sent_tokenize(str(review))
    
    for sentence in sentences:
        aspects = get_aspect(sentence)
        if aspects:
            # Cümleyi TF-IDF ile dönüştür
            tfidf_matrix = tfidf_vectorizer.transform([sentence])
            # Orijinal incelemenin sayısal özelliklerini al
            num_matrix = X_numeric_scaled[idx].reshape(1, -1)
            
            # İkisini birleştir
            combined_matrix = scipy.sparse.hstack([tfidf_matrix, scipy.sparse.csr_matrix(num_matrix)])
            
            # Eğitilmiş modelimizle tahmin yap
            pred_prob = lr_model.predict(combined_matrix)[0]
            
            class_map = {0: 'Kötü', 1: 'Nötr', 2: 'İyi'}
            pred_label = class_map[pred_prob]
            
            for aspect in aspects:
                aspect_results.append({
                    'review_id': idx,
                    'aspect': aspect,
                    'sentence': sentence,
                    'sentiment': pred_label
                })

df_aspects = pd.DataFrame(aspect_results)
print(f"Toplam {len(df_aspects)} adet Varlık-Cümle eşleşmesi bulundu.")
df_aspects.head(10)


## 7.4 Sonuçların Görselleştirilmesi

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df_aspects, x='aspect', hue='sentiment', palette={'Kötü':'red', 'Nötr':'gray', 'İyi':'green'})
plt.title('Varlıklara (Aspects) Göre Duygu Dağılımı')
plt.xlabel('Varlık (Aspect)')
plt.ylabel('Cümle Sayısı')
plt.legend(title='Tahmin')
plt.tight_layout()
import os
os.makedirs('results', exist_ok=True)
plt.savefig('results/aspect_based_sentiment.png')
plt.show()
